# C3 model v2 review: diagnosis, data audit and benchmark

This notebook replays the 20 October 2026 investigation from saved evidence. It needs no GPU.

1. **Diagnosis**: the same v1 adapter, same validation screens, app input versus training input.
2. **Data audit**: how many synthetic mutations are visible at training resolution, and a label sample grid.
3. **Benchmark**: base model, v1 and v2 through the production backend (`contract_benchmark.py`).

What to conclude is written under each output.

In [ ]:
import json, glob, os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
EV = Path('..') / 'evidence' / '11_model_v2'
def load(p):
    return json.load(open(p, encoding='utf-8'))

## 1. Diagnosis: the prompt and resolution are part of the model

In [ ]:
def score(records):
    tp = sum(r['gold_has'] and r['pred_has'] is True for r in records)
    fp = sum((not r['gold_has']) and r['pred_has'] is True for r in records)
    fn = sum(r['gold_has'] and r['pred_has'] is not True for r in records)
    return dict(n=len(records), caught=f"{tp}/{tp+fn}", false_alarms=fp)
app = load(EV/'diagnosis'/'v1_adapter_app_input_val18.json')
train = load(EV/'diagnosis'/'v1_adapter_training_input_val60.json')[:len(app)]
base = load(EV/'diagnosis'/'base_app_input_val26.json')
print('v1 adapter, old app input     ', score(app))
print('v1 adapter, training input    ', score(train))
print('base model, old app input     ', score(base))

**Look at:** the `caught` column. **Conclude:** the adapter learned something, but the app's long system prompt and 3.5x larger images pushed it back to the base model's habit of answering "no defect". Every adapter now ships `inference_contract.json`, which the backend enforces.

## 2. Data audit

In [ ]:
vis = load(EV/'data_audit'/'mut_visibility.json')
vals = {k: v for k, v in vis.items() if v is not None}
by = {}
for p, v in vals.items():
    by.setdefault(p.split('_mut_')[1].split('.')[0], []).append(v * 100)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist([by[k] for k in sorted(by)], bins=[0, .05, .1, .2, .5, 1, 2, 5, 10, 50], label=sorted(by), stacked=True)
ax.set_xscale('symlog', linthresh=0.1); ax.axvline(0.2, color='k', ls='--')
ax.set_xlabel('% of pixels changed at 262,144-px training resolution'); ax.set_ylabel('mutations'); ax.legend()
print('invisible (<0.2% pixels):', sum(v < 0.002 for v in vals.values()), 'of', len(vals))

**Look at:** the bars left of the dashed line. **Conclude:** those mutations look clean to the model but are labelled defective. v2 removes them from train and validation (`dataset_v2.py`).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 20))
for ax, name in zip(axes, ['owl_grid.png', 'rico_grid.png']):
    ax.imshow(Image.open(EV/'data_audit'/name)); ax.set_title(name); ax.axis('off')

**Look at:** each label versus what the screen actually shows. **Conclude:** defect-vs-clean is trustworthy; defect type and box are often wrong (for example `nullnull` labelled as overlap). Report binary detection as the headline metric.

## 3. Benchmark through the production backend

In [ ]:
rows = []
for f in sorted(glob.glob(str(EV/'benchmarks'/'*_scores.json'))):
    d = load(f)
    rows.append((os.path.basename(f).replace('_scores.json', ''), len(d['records']), d['auc'], d.get('auc_95ci', '-'), d['best_threshold']['threshold']))
print(f"{'run (P(defect) scoring)':34} {'n':>4} {'AUC':>6} {'95% CI':>14} {'F1-best thr':>11}")
for r in rows:
    print(f"{r[0]:34} {r[1]:>4} {r[2]:>6.3f} {str(r[3]):>14} {r[4]:>11.3f}")
e2e = load(EV/'benchmarks'/'final_test316_end_to_end.json')['summary']
print()
print('Shipped model, end to end on test:', {k: e2e[k] for k in ('precision', 'recall', 'f1', 'accuracy', 'type_accuracy_on_defects', 'median_latency_ms')})
print('McNemar:', load(EV/'benchmarks'/'mcnemar_test316.json'))

**Look at:** `*_val291` rows chose the shipped model (highest AUC) and its threshold; `*_test316` rows were run once afterwards. `base_v2prompt_*` uses the identical prompt and resolution with no adapter, so the gap is what fine-tuning added. **Conclude:** v1 beats the base model clearly (McNemar p = 0.0001); the cleaner v2 retrain does not beat v1, so labels are now the bottleneck.

In [ ]:
p = EV/'benchmarks'/'final_test316_end_to_end.json'
if p.exists():
    s = load(p)['summary']
    names = sorted(s['per_class'])
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(names, [s['per_class'][k]['binary_correct'] / s['per_class'][k]['n'] for k in names])
    ax.set_ylabel('share judged correctly (defect vs clean)'); ax.set_ylim(0, 1); plt.xticks(rotation=20)

**Conclude:** per-class bars show which defect kinds the 2B model sees at 384-pixel width and which it misses.

## 4. Dataset v3 (21 October 2026)

Regenerated synthetic defects, app-grouped splits and relabelled OwlEye val/test. Details: `evidence/DATASET_V3_DESIGN.md`.

In [ ]:
EV3 = Path('..') / 'evidence' / '12_dataset_v3'
rep = load(EV3 / 'dataset_v3_report.json')
for split, counts in rep['splits'].items():
    print(split, sum(counts.values()), counts)
print('layout text shortened:', rep.get('layout_shortened'), '| OwlEye withheld from train:', rep.get('owleye_withheld_no_labelled_bugs'))
for s in ('train', 'val', 'test'):
    m = load(EV3 / f'mutation_report_{s}.json'); print(s, 'mutations kept', m['kept'], m['by_type'], 'rejected', m['rejected'])

**Look at:** the clean/defect balance per split, the mutation rejections by reason, and that OwlEye is withheld from train. **Conclude:** every synthetic defect left is visible at training resolution, and no app crosses splits.

In [ ]:
import random
from PIL import ImageDraw
prov = [r for r in load(EV3 / 'dataset_v3_provenance.json') if r['split'] == 'test' and r['label']['defect_type'] != 'none']
random.Random(0).shuffle(prov)
fig, axes = plt.subplots(1, 6, figsize=(18, 6))
for ax, r in zip(axes, prov[:6]):
    im = Image.open(r['image']).convert('RGB'); W, H = im.size; d = ImageDraw.Draw(im)
    y1, x1, y2, x2 = r['label']['bounding_box']; d.rectangle([x1*W/1000, y1*H/1000, x2*W/1000, y2*H/1000], outline='red', width=6)
    ax.imshow(im); ax.set_title(f"{r['source']}\n{r['label']['defect_type']}", fontsize=9); ax.axis('off')

**Look at:** the red box sits on the defect for each type and source. **Conclude:** after the coordinate fix, boxes match what is drawn.

In [ ]:
labels = load(EV3 / 'owleye_labels_v3.json')
from collections import Counter
print(Counter((v['split'], v['defect_type'] or v['status']) for v in labels.values()))
print('human verified:', sum(v.get('human_verified') for v in labels.values()), 'of', len(labels))

**Conclude:** labels are model-assisted until `human_verified` is set. Check `review_sheets/review_test.csv` before reporting defect-type accuracy as ground truth.